# RSNA Knee — submission skeleton

Smoke test for our submission pipeline. It does **no modelling**: it writes the
benchmark 0.5 for every study, exactly like `sample_submission.csv`.

Its only job is to prove the submission mechanics before anything depends on them:

1. the competition data is mounted where we think it is,
2. `submission.csv` is written to the working directory under the right name and
   the right schema,
3. the whole thing runs with internet disabled.

Nothing else is attached — no label table, no weights — so a failure here can only
be about the mechanics themselves.

Expected score: **0.500**.

Pushed from the repo by `python -m scripts.kaggle_push`. Do not edit in the Kaggle
UI: the next push overwrites it, and a browser edit is invisible to the other
person.


In [ ]:
# Replaced at push time by scripts/kaggle_push.py: ties this run to a commit.
RUN_STAMP = "dev"
print(f"run stamp: {RUN_STAMP}")


In [ ]:
import time
from pathlib import Path

import pandas as pd

T0 = time.time()

TARGETS = ["ACL", "MCL", "Medial Meniscus", "Lateral Meniscus", "Medial OA",
           "Lateral OA", "PF OA", "Effusion", "Synovitis", "Baker's",
           "Contusion", "Fracture"]


def log(msg):
    print(f"[{time.time() - T0:6.1f}s] {msg}", flush=True)


def find_root():
    """Locate the competition mount.

    Same candidate order as the public baseline notebook, so that whatever works
    here works there. On Kaggle the mount is /kaggle/input/<competition-slug>;
    locally it is data/raw.
    """
    for candidate in [Path("/kaggle/input/rsna-knee-abnormality-detection"),
                      Path("/kaggle/input/competitions/rsna-knee-abnormality-detection"),
                      Path("data/raw"),
                      Path("data")]:
        if (candidate / "test.csv").is_file():
            return candidate
    raise FileNotFoundError("competition mount not found")


ROOT = find_root()
log(f"competition root: {ROOT}")
for name in ["test.csv", "test_series.csv", "sample_submission.csv"]:
    log(f"  {name}: {'present' if (ROOT / name).is_file() else 'MISSING'}")
log(f"  test_series/: {'present' if (ROOT / 'test_series').is_dir() else 'MISSING'}")

In [ ]:
test = pd.read_csv(ROOT / "test.csv", dtype={"StudyInstanceUID": str})
log(f"test studies: {len(test)}")

submission = pd.DataFrame({"StudyInstanceUID": test["StudyInstanceUID"]})
for target in TARGETS:
    submission[target] = 0.5

# The file must be named exactly this, in the working directory.
submission.to_csv("submission.csv", index=False)

log(f"wrote submission.csv: {submission.shape}")
print(submission.head(3).to_string(index=False))

expected = ["StudyInstanceUID"] + TARGETS
assert list(submission.columns) == expected, "submission schema drift"
assert submission["StudyInstanceUID"].is_unique, "duplicate study ids"
assert submission[TARGETS].notna().all().all(), "missing predictions"
log("schema checks passed")